# 🚀 MyUPI Voice + Action Assistant on Google Colab (GPU)

This notebook runs the complete **MyUPI Voice Assistant** on a free Google Colab T4 GPU with full **Voice-In (ASR)** and **Voice-Out (TTS)** enabled using Hugging Face models!

### Step 1: Check GPU
Ensure you are connected to a GPU runtime (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Install System and Python Dependencies

In [ ]:
!apt-get install -y ffmpeg
!pip install -r requirements.txt
!pip install faster-whisper TTS pycloudflared

### Step 3: Pre-download Hugging Face Voice Models (ASR & TTS)
- **ASR**: Faster-Whisper (CUDA accelerated Whisper from HuggingFace)
- **TTS**: Coqui VITS (High quality, low latency text-to-speech)

In [ ]:
from faster_whisper import WhisperModel
print("📥 Downloading Faster-Whisper ASR model from Hugging Face...")
asr_model = WhisperModel('small', device='cuda', compute_type='float16', download_root='models/whisper')
print("✅ ASR model ready on GPU!")

from TTS.api import TTS
print("📥 Downloading Coqui VITS TTS model...")
tts_model = TTS('tts_models/en/ljspeech/vits', progress_bar=True)
print("✅ TTS model ready!")

### Step 4: Configure `config/config.yaml` for Local GPU Voice
This sets ASR to `faster_whisper` on `cuda`, and TTS to `coqui`.

In [ ]:
import yaml

with open('config/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# Set ASR to faster_whisper on CUDA
cfg['asr']['provider'] = 'faster_whisper'
cfg['asr']['model_authoritative'] = 'small'
cfg['asr']['device'] = 'cuda'
cfg['asr']['compute_type'] = 'float16'

# Set TTS to coqui
cfg['tts']['provider'] = 'coqui'
cfg['tts']['model'] = 'tts_models/en/ljspeech/vits'

with open('config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("✅ config.yaml updated for Colab GPU Voice pipeline!")

### Step 5: Launch Server with Public HTTPS URL (Cloudflare Tunnel)
Click the generated `trycloudflare.com` public URL to open the MyUPI Web App in your browser!

In [ ]:
import subprocess
import time
from pycloudflared import try_cloudflare

# Start Cloudflare tunnel on port 8000
tunnel_url = try_cloudflare(port=8000)
print(f"\n🎉 YOUR PUBLIC APP URL: {tunnel_url.tunnel}\n")
print("Open this link in your browser to test Voice-In and Voice-Out!\n")

# Launch Uvicorn
!uvicorn app.main:app --host 0.0.0.0 --port 8000